In [0]:
kafka_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka-instance-bipin-data-engineering-project-1.j.aivencloud.com:10670")
        .option("subscribe", "user_activity_data_gen")
        .option("startingOffsets", "earliest")
        .option("kafka.security.protocol", "SSL")
        .option(
            "kafka.ssl.truststore.type",
            "PEM"
        )
        .option(
            "kafka.ssl.keystore.type",
            "PEM"
        )
        .option(
            "kafka.ssl.truststore.certificates",
            open("/Volumes/kafka/default/certs/ca.pem").read()
        )
        .option(
            "kafka.ssl.keystore.certificate.chain",
            open("/Volumes/kafka/default/certs/service.cert").read()
        )
        .option(
            "kafka.ssl.keystore.key",
            open("/Volumes/kafka/default/certs/service.key").read()
        )
        .load()
)

In [0]:
avro_schema = """
{
  "namespace": "data.gen.avro",
  "name": "user_activity",
  "type": "record",
  "fields": [
    {
      "name": "time_utc",
      "type": "long",
      "examples": [
        1747052327
      ]
    },
    {
      "name": "action_id",
      "type": "string",
      "examples": "page9001"
    },
    {
      "name": "action",
      "type": "string",
      "examples": [
        "cancel",
        "confirm",
        "next"
      ]
    },
    {
      "name": "page",
      "type": "string",
      "examples": [
        "/app/checkout",
        "/app/trends",
        "/app/most-popular"
      ]
    },
    {
      "name": "section",
      "type": "string",
      "examples": [
        "quick-view",
        "add-to-basket",
        "remove-from-basket",
        "remove-from-basket",
        "sale"
      ]
    },
    {
      "name": "country_code",
      "type": "string",
      "examples": [
        "ie",
        "GB",
        "USA",
        "DE",
        "PL",
        "CA",
        "BE"
      ]
    },
    {
      "name": "user_id",
      "type": "string",
      "examples": [
        "user@email.com",
        "anonymous_109827",
        "logged_in@email.com"
      ]
    }
  ]
}
"""

In [0]:
from pyspark.sql.functions import expr, col
from pyspark.sql.avro.functions import from_avro

decoded_df = (
    kafka_df
    .withColumn(
        "payload",
        expr("substring(value, 6)")
    )
    .select(
        from_avro(
            col("payload"),
            avro_schema,
            {"mode":"PERMISSIVE"}
        ).alias("data"),'offset','timestamp','topic'
    )
)

In [0]:
try:
        query = (
        decoded_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", bronze_checkpoint_path)
        .trigger(availableNow=True)
        .table("kafka.default.user_activity_data_gen_bronze"))

        query.awaitTermination()

        print("Batch completed. Sleeping...")
        # print("Visual refresh started...")
        # dbutils.notebook.run("/Workspace/Users/imbipin8858@gmail.com/dashboard",300)
        # print("Visual Refreshed!")


except Exception as e:
    print(f"Error: {e}")

In [0]:
%skip
import time
bronze_path = "/Volumes/kafka/default/bronze/data/user_activity_data_gen"
bronze_checkpoint_path = "/Volumes/kafka/default/bronze/checkpoint/user_activity_data_gen"
while True:
    try:
        query = (
        decoded_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", bronze_checkpoint_path)
        .trigger(availableNow=True)
        .table("kafka.default.user_activity_data_gen"))

        query.awaitTermination()

        print("Batch completed. Sleeping...")
        # print("Visual refresh started...")
        # dbutils.notebook.run("/Workspace/Users/imbipin8858@gmail.com/dashboard",300)
        # print("Visual Refreshed!")


    except Exception as e:
        print(f"Error: {e}")

    time.sleep(10)

In [0]:
%skip
bronze_checkpoint_path = "/Volumes/kafka/default/tmp/checkpoints/user_activity_data_gen"

bronze_query = (
    decoded_df.writeStream
        .format("delta")
        .outputMode("append")
        .option('mode','permissive')
        .option(
            "checkpointLocation",
            bronze_checkpoint_path
        )
        # .trigger(availableNow=True)
        .trigger(processingTime="10 seconds")
        .table("kafka.default.user_activity_data_gen")
)

In [0]:
%skip
    %sql
select data.* From kafka.default.user_activity_data_gen order by data.time_utc;